# PAATRA Step 3b — Part 4/5: Train Config C (C_10K_paatra)

Trains the **C_10K_paatra** student. Loads the chunks and vocabulary artifacts saved by `03_scaled_setup.ipynb`, runs distillation, and saves the student bundle to Drive.

**Prerequisite:** run `03_scaled_setup.ipynb` first.

The recovered experiment reached step 14,000 of 15,000 before the Colab session stopped. This notebook keeps the full 15,000-step target but should save intermediate checkpoints when used for reproduction.


In [ ]:
!pip install -q torch transformers accelerate datasets

In [ ]:
import torch, math, time, os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEACHER_ID = "Qwen/Qwen2.5-0.5B"
PRESET = "FULL"

if PRESET == "FULL":
    SEQ_LEN, BATCH_SIZE, NUM_TRAIN_STEPS, NUM_CORPUS_SAMPLES = 512, 4, 15000, 80_000
elif PRESET == "FAST":
    SEQ_LEN, BATCH_SIZE, NUM_TRAIN_STEPS, NUM_CORPUS_SAMPLES = 256, 8, 8000, 50_000

LEARNING_RATE = 3e-4
WARMUP_STEPS = 500
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.7
CHECKPOINT_EVERY = 1000

print(f'Device: {DEVICE} | Preset: {PRESET}')


## 1. Mount Drive and load artifacts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/paatra'
os.makedirs(f'{DRIVE_DIR}/students', exist_ok=True)
NAME = "C_10K_paatra"

meta = torch.load(f'{DRIVE_DIR}/meta.pt', weights_only=False)
chunks = torch.load(f'{DRIVE_DIR}/chunks.pt', weights_only=False)
vocabs = torch.load(f'{DRIVE_DIR}/vocabs.pt', weights_only=False)
assert meta['SEQ_LEN'] == SEQ_LEN
assert meta['NUM_TRAIN_STEPS'] == NUM_TRAIN_STEPS

s2t = vocabs[NAME]['s2t']
t2s = vocabs[NAME]['t2s']
config_dict = vocabs[NAME]['config']
print(f'Vocab: {len(s2t):,} | Config: {config_dict}')


## 2. Load teacher

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GPT2Config, GPT2LMHeadModel

tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, torch_dtype=torch.float16, device_map=DEVICE,
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False


## 3. Define student and dataset

In [ ]:
def create_student(config_dict, vocab_size):
    config = GPT2Config(
        vocab_size=vocab_size,
        n_embd=config_dict["n_embd"],
        n_layer=config_dict["n_layer"],
        n_head=config_dict["n_head"],
        n_inner=4 * config_dict["n_embd"],
        activation_function="gelu_new",
        resid_pdrop=0.1, embd_pdrop=0.1, attn_pdrop=0.1,
        n_positions=SEQ_LEN,
        bos_token_id=None, eos_token_id=None, pad_token_id=None,
    )
    return GPT2LMHeadModel(config)

def cosine_lr(step, warmup, total, base_lr):
    if step < warmup:
        return base_lr * step / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

class KDDataset(Dataset):
    def __init__(self, chunks, t2s):
        self.chunks = chunks
        self.t2s = t2s
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, idx):
        teacher_ids = self.chunks[idx].tolist()
        student_ids = [self.t2s.get(tid, 0) for tid in teacher_ids]
        loss_mask = [1.0 if tid in self.t2s else 0.0 for tid in teacher_ids]
        return {
            "teacher_ids": torch.tensor(teacher_ids, dtype=torch.long),
            "student_ids": torch.tensor(student_ids, dtype=torch.long),
            "loss_mask": torch.tensor(loss_mask, dtype=torch.float),
        }


## 4. Train with intermediate checkpointing

In [ ]:
student_vocab_size = len(s2t)
s2t_tensor = torch.tensor(s2t, dtype=torch.long, device=DEVICE)
student = create_student(config_dict, student_vocab_size).to(DEVICE)
total_params = sum(p.numel() for p in student.parameters())
emb_params = student_vocab_size * config_dict['n_embd']
print(f'Vocab: {student_vocab_size:,} | Total params: {total_params:,}')
print(f'Emb: {emb_params/total_params*100:.1f}% | Trans: {(1-emb_params/total_params)*100:.1f}%')

dataset = KDDataset(chunks, t2s)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

student.train()
step = 0
running_loss = running_kd = running_ce = 0.0
loss_history = []
start = time.time()

while step < NUM_TRAIN_STEPS:
    for batch in dataloader:
        if step >= NUM_TRAIN_STEPS:
            break
        lr = cosine_lr(step, WARMUP_STEPS, NUM_TRAIN_STEPS, LEARNING_RATE)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        teacher_ids = batch['teacher_ids'].to(DEVICE)
        student_ids = batch['student_ids'].to(DEVICE)
        loss_mask = batch['loss_mask'].to(DEVICE)
        assert student_ids.min().item() >= 0
        assert student_ids.max().item() < student_vocab_size

        with torch.no_grad():
            t_logits = teacher(input_ids=teacher_ids).logits.float()
            t_subset = t_logits[:, :-1, :].index_select(2, s2t_tensor)
        s_logits = student(input_ids=student_ids).logits[:, :-1, :]
        labels = student_ids[:, 1:]
        mask = loss_mask[:, 1:]
        mask_sum = mask.sum().clamp(min=1)

        t_probs = F.softmax(t_subset / KD_TEMPERATURE, dim=-1)
        s_log_probs = F.log_softmax(s_logits / KD_TEMPERATURE, dim=-1)
        kd_per_token = F.kl_div(s_log_probs, t_probs, reduction='none').sum(-1)
        kd_loss = (kd_per_token * mask).sum() / mask_sum * (KD_TEMPERATURE ** 2)
        ce_per_token = F.cross_entropy(
            s_logits.reshape(-1, s_logits.size(-1)), labels.reshape(-1), reduction='none'
        ).reshape(labels.shape)
        ce_loss = (ce_per_token * mask).sum() / mask_sum
        loss = KD_ALPHA * kd_loss + (1 - KD_ALPHA) * ce_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item(); running_kd += kd_loss.item(); running_ce += ce_loss.item(); step += 1
        if step % CHECKPOINT_EVERY == 0:
            avg = running_loss / CHECKPOINT_EVERY
            avg_kd = running_kd / CHECKPOINT_EVERY
            avg_ce = running_ce / CHECKPOINT_EVERY
            elapsed = time.time() - start
            loss_history.append({'step': step, 'loss': avg, 'kd': avg_kd, 'ce': avg_ce})
            print(f'Step {step:>6}/{NUM_TRAIN_STEPS} | Loss: {avg:.3f} | KD: {avg_kd:.3f} | CE: {avg_ce:.3f} | LR: {lr:.1e} | {elapsed/60:.0f}m')
            bundle = {
                'name': NAME, 'state_dict': student.state_dict(),
                'model_config': student.config.to_dict(), 'config': config_dict,
                's2t': s2t, 't2s': t2s, 'losses': loss_history,
                'preset': PRESET, 'seq_len': SEQ_LEN, 'training_step': step,
                'optimizer_state_dict': optimizer.state_dict(),
            }
            torch.save(bundle, f'{DRIVE_DIR}/students/{NAME}_step{step}.pt')
            running_loss = running_kd = running_ce = 0.0


## 5. Save final bundle

In [ ]:
student.eval()
bundle = {
    'name': NAME, 'state_dict': student.state_dict(),
    'model_config': student.config.to_dict(), 'config': config_dict,
    's2t': s2t, 't2s': t2s, 'losses': loss_history,
    'preset': PRESET, 'seq_len': SEQ_LEN,
    'num_train_steps': NUM_TRAIN_STEPS, 'training_step': step,
}
out_path = f'{DRIVE_DIR}/students/{NAME}.pt'
torch.save(bundle, out_path)
print(f'Saved: {out_path}')
